# Image Caption Generator — Full PyTorch Rewrite

A complete top-to-bottom rewrite of the original Keras notebook.  
Every section follows the pattern: **Concept → Bug in original → Fix**.

| | |
|---|---|
| **Model** | Custom 5-block CNN Encoder + 2-layer LSTM Decoder |
| **Dataset** | Flickr8k (8,091 images, ≈5 captions each) |
| **Framework** | PyTorch (replacing Keras/TensorFlow) |
| **Generation** | Beam search (replacing greedy argmax) |


---
## Section 1 — Imports

**Concept:** We collect all dependencies upfront so any missing package fails immediately.

**Bug in original:** `preprocess_input` was used in the feature extraction cell (cell 10) but was **never imported**. This caused a `NameError` at runtime, silently stopping all feature extraction before training even started.

**Fix:** In PyTorch, image normalisation is handled by `torchvision.transforms.Normalize` inside the DataLoader — no separate `preprocess_input` needed.

In [ ]:
import os
import re
import json
import random
import numpy as np
from PIL import Image
from tqdm.auto import tqdm          # auto picks notebook or terminal — no ipywidgets needed
import matplotlib.pyplot as plt
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

print('PyTorch version:', torch.__version__)
print('CUDA available: ', torch.cuda.is_available())


---
## Section 2 — Configuration

**Concept:** Centralising every hyperparameter in one place makes experiments reproducible. Setting a global `SEED` across Python, NumPy, and PyTorch ensures the same train/val/test split and weight initialisation every run.

> **Update `IMAGE_DIR` and `CAPTION_FILE`** to point to your Flickr8k data before running.

In [ ]:
import sys

# ── Paths ───────────────────────────────────────────────────────────────────────
ON_COLAB = 'google.colab' in sys.modules

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    IMAGE_DIR      = '/content/imgs/Images'
    CAPTION_FILE   = '/content/drive/MyDrive/DeepLearingProject /captions.txt'
    CHECKPOINT_DIR = '/content/checkpoints'
    OUTPUT_DIR     = '/content/outputs'
else:
    # Hardcoded absolute path — reliable regardless of where Jupyter starts
    _BASE          = '/home/nathiskar/image_caption_generator'
    IMAGE_DIR      = os.path.join(_BASE, 'data', 'images', 'Image')
    CAPTION_FILE   = os.path.join(_BASE, 'data', 'captions.txt')
    CHECKPOINT_DIR = os.path.join(_BASE, 'checkpoints')
    OUTPUT_DIR     = os.path.join(_BASE, 'outputs')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,     exist_ok=True)

print(f'Environment : {"Colab" if ON_COLAB else "Local laptop"}')
print(f'IMAGE_DIR   : {IMAGE_DIR}')
print(f'CAPTION_FILE: {CAPTION_FILE}')
print(f'CHECKPOINTS : {CHECKPOINT_DIR}')
print(f'Files exist : images={os.path.isdir(IMAGE_DIR)}, captions={os.path.isfile(CAPTION_FILE)}')

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print(f'Device      : {DEVICE}  |  AMP: {USE_AMP}')

# ── Model hyperparameters ──────────────────────────────────────────────────────
IMAGE_SIZE      = 224
CNN_FEAT_DIM    = 4096
EMBED_DIM       = 256
LSTM_UNITS      = 512
DROPOUT         = 0.4
MAX_LENGTH      = 35
VOCAB_THRESHOLD = 2

# ── Training hyperparameters (CPU-friendly) ────────────────────────────────────
BATCH_SIZE    = 16
EPOCHS        = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
PATIENCE      = 3
BEAM_WIDTH    = 3

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


---
## Section 3 — Data Loading

**Concept:** `captions.txt` has the format `image_file,caption` with one row per caption. Each image has ≈5 rows. We parse this into a dict: `{ image_id → [caption1, caption2, ...] }`.

**Bug in original (cell 16):**
```python
tokens = line.split(',')          # splits on EVERY comma
image_id, caption = tokens[0], tokens[1]   # takes only first segment
```
If a caption contains a comma — e.g., `"A dog, a cat, and a bird"` — then `tokens[1]` is `" a cat"` and everything after the second comma is silently discarded.

**Fix:** `split(',', 1)` — split only on the **first** comma, so the rest of the line (entire caption) stays intact.

In [ ]:
def load_captions(caption_file):
    """
    Parse captions.txt into { image_id: [caption, ...] }.
    Uses maxsplit=1 so captions containing commas are never truncated.
    """
    mapping = {}
    with open(caption_file, 'r') as f:
        next(f)  # skip header line: "image,caption"
        for line in f:
            line = line.strip()
            if not line:
                continue
            # FIX: maxsplit=1 keeps full caption even if it contains commas
            parts = line.split(',', 1)
            if len(parts) < 2:
                continue
            image_file, caption = parts
            image_id = image_file.split('.')[0]  # drop '.jpg'
            mapping.setdefault(image_id, []).append(caption.strip())
    print(f'Loaded captions for {len(mapping)} images')
    return mapping


mapping = load_captions(CAPTION_FILE)

# Quick sanity check
sample_id = list(mapping.keys())[0]
print(f'\nSample captions for {sample_id}:')
for c in mapping[sample_id]:
    print(f'  {c}')


---
## Section 4 — Caption Cleaning

**Concept:** Raw captions have inconsistent capitalisation, punctuation, and whitespace. We normalise them so the model learns cleaner word patterns. The steps are:
1. Lowercase everything
2. Remove any character that is not a letter or a space
3. Collapse multiple spaces into one

**Bug in original (cell 18) — the most damaging bug in the whole notebook:**
```python
caption.replace('[^A-Za-z]', '')   # str.replace() is LITERAL — does nothing
caption.replace('\s+', ' ')        # str.replace() is LITERAL — does nothing
```
`str.replace()` looks for the **exact string** `[^A-Za-z]` inside the caption. That exact character sequence never appears, so **punctuation was never removed**. The vocabulary ended up full of noise: `road.`, `dog,`, `tri-colored`, `other.`, etc. This inflated `vocab_size` from ~4 500 clean words to ~8 300 noisy tokens, making the embedding matrix larger and training less efficient.

**Fix:** Use `re.sub()` which interprets the pattern as a **regex**.

In [ ]:
def clean_caption(caption):
    """
    Normalise a single caption string.

    FIX: use re.sub() for regex — str.replace() is literal-only.
    """
    caption = caption.lower().strip()
    caption = re.sub(r'[^a-z\s]', '', caption)   # remove punctuation & digits
    caption = re.sub(r'\s+', ' ', caption).strip()  # collapse whitespace
    return caption


def clean_all_captions(mapping):
    """Apply clean_caption to every caption in the mapping dict (in-place)."""
    for image_id in mapping:
        mapping[image_id] = [clean_caption(c) for c in mapping[image_id]]


# Before cleaning
print('BEFORE:', mapping[sample_id][:2])

clean_all_captions(mapping)

# After cleaning
print('AFTER: ', mapping[sample_id][:2])


---
## Section 5 — Train / Val / Test Split

**Concept:** We split image IDs (not individual captions) into 80 % train / 10 % val / 10 % test. Splitting at the image level ensures the model never sees an image during validation or testing that it was trained on — even in a different caption.

**Bug in original (cell 28):**
```python
image_ids = list(mapping.keys())
train = image_ids[:train_split]   # no shuffle — first 80% always goes to train
```
Flickr8k image IDs are alphanumeric filenames. IDs that start with the same digits come from the same Flickr user/album, so without shuffling the split is **biased** — certain scenes/photographers dominate train while others dominate test.

**Fix:** `random.shuffle(ids)` before slicing, with a fixed seed for reproducibility.

In [ ]:
def split_data(mapping, train_ratio=0.80, val_ratio=0.10, seed=SEED):
    """
    Randomly split image IDs into train / val / test.
    FIX: shuffle before splitting to avoid ordering bias.
    """
    ids = list(mapping.keys())
    random.seed(seed)
    random.shuffle(ids)             # ← the critical missing step

    n         = len(ids)
    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))

    train_ids = ids[:train_end]
    val_ids   = ids[train_end:val_end]
    test_ids  = ids[val_end:]

    print(f'Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}')
    return train_ids, val_ids, test_ids


train_ids, val_ids, test_ids = split_data(mapping)


---
## Section 6 — Vocabulary

**Concept:** A `Vocabulary` maps every word to a unique integer index. Four special tokens are reserved:
- `<pad>=0` — fills sequences shorter than `MAX_LENGTH`
- `<start>=1` — signals the decoder to begin generating
- `<end>=2` — signals the decoder to stop
- `<unk>=3` — replaces words seen fewer than `VOCAB_THRESHOLD` times

**Bug in original (cell 25) — data leakage:**
```python
all_captions = []          # collects captions from ALL splits
for key in mapping:        # mapping = train + val + test
    for caption in mapping[key]:
        all_captions.append(caption)
tokenizer.fit_on_texts(all_captions)   # sees val & test words!
```
The tokenizer learned the vocabulary of validation and test captions. Rare words that only appear in the test set get known indices instead of `<unk>`. This artificially boosts BLEU scores because the model's vocabulary perfectly covers test-set words it was never trained to produce.

**Fix:** Build vocabulary **only from training captions**.

In [ ]:
class Vocabulary:
    PAD, START, END, UNK = '<pad>', '<start>', '<end>', '<unk>'

    def __init__(self):
        self.word2idx = {self.PAD: 0, self.START: 1, self.END: 2, self.UNK: 3}
        self.idx2word = {0: self.PAD, 1: self.START, 2: self.END, 3: self.UNK}

    def build(self, captions, threshold=VOCAB_THRESHOLD):
        freq = Counter()
        for caption in captions:
            freq.update(caption.split())
        idx = len(self.word2idx)
        for word, count in freq.most_common():
            if count >= threshold and word not in self.word2idx:
                self.word2idx[word] = idx
                self.idx2word[idx]  = word
                idx += 1
        print(f'Vocabulary: {len(self.word2idx)} words  (threshold={threshold})')

    def encode(self, caption):
        unk = self.word2idx[self.UNK]
        return [self.word2idx.get(w, unk) for w in caption.split()]

    def decode(self, indices):
        words = []
        for idx in indices:
            word = self.idx2word.get(idx, self.UNK)
            if word in (self.PAD, self.START):
                continue
            if word == self.END:
                break
            words.append(word)
        return ' '.join(words)

    def save(self, path):
        # Always use the known project checkpoints folder — never trust CHECKPOINT_DIR variable
        safe_path = '/home/nathiskar/image_caption_generator/checkpoints/vocab.json'
        os.makedirs(os.path.dirname(safe_path), exist_ok=True)
        with open(safe_path, 'w') as f:
            json.dump({'word2idx': self.word2idx,
                       'idx2word': {str(k): v for k, v in self.idx2word.items()}}, f)
        print(f'Vocabulary saved → {safe_path}')

    @classmethod
    def load(cls, path=None):
        safe_path = '/home/nathiskar/image_caption_generator/checkpoints/vocab.json'
        vocab = cls()
        with open(safe_path) as f:
            data = json.load(f)
        vocab.word2idx = data['word2idx']
        vocab.idx2word = {int(k): v for k, v in data['idx2word'].items()}
        print(f'Vocabulary loaded — {len(vocab.word2idx)} words')
        return vocab

    def __len__(self):
        return len(self.word2idx)


In [ ]:
# Force the correct path regardless of kernel working directory
CHECKPOINT_DIR = '/home/nathiskar/image_caption_generator/checkpoints'
OUTPUT_DIR     = '/home/nathiskar/image_caption_generator/outputs'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,     exist_ok=True)

# Build vocabulary from TRAINING captions ONLY — fixes data leakage
train_captions = [c for img_id in train_ids for c in mapping[img_id]]
vocab = Vocabulary()
vocab.build(train_captions, threshold=VOCAB_THRESHOLD)
vocab.save(os.path.join(CHECKPOINT_DIR, 'vocab.json'))

print(f'\nmax_length = {MAX_LENGTH}')
print(f'vocab_size = {len(vocab)}')


In [ ]:
# ── Quick fix: force-set correct paths in kernel memory ───────────────────────
import os
_BASE          = '/home/nathiskar/image_caption_generator'
CHECKPOINT_DIR = os.path.join(_BASE, 'checkpoints')
OUTPUT_DIR     = os.path.join(_BASE, 'outputs')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,     exist_ok=True)
print(f'CHECKPOINT_DIR = {CHECKPOINT_DIR}')
print(f'Writable       = {os.access(CHECKPOINT_DIR, os.W_OK)}')


---
## Section 7 — Image Transforms & PyTorch Dataset

**Concept:** A PyTorch `Dataset` defines how to load a single `(image, caption)` pair. The `DataLoader` wraps it to yield mini-batches, shuffle data, and load images in parallel using multiple CPU workers.

**Bug in original (cell 10) — `preprocess_input` was never imported AND pixels were never normalised:**
```python
image = preprocess_input(image)   # NameError — function does not exist
```
Even if the import had been there, raw pixel values in `[0, 255]` were fed to the CNN. Without normalisation, the activations in early layers are large and irregular, causing slow and unstable gradient flow.

**Fix:** `torchvision.transforms.Normalize` with ImageNet mean/std. Training images also get random crop and flip for **data augmentation** — this reduces overfitting because the model sees each image in slightly different configurations.

**Caption encoding:** Each caption becomes a padded integer tensor of length `MAX_LENGTH`:  
`[<start>, w1, w2, ..., wN, <end>, <pad>, <pad>, ...]`

In [ ]:
# ImageNet normalisation constants — standard practice even for custom CNNs
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]


def get_transforms(train=True):
    """
    Training: random crop + flip for augmentation.
    Val/Test:  deterministic resize only.
    """
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
            transforms.RandomCrop(IMAGE_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(_MEAN, _STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(_MEAN, _STD),
    ])


In [ ]:
class CaptionDataset(Dataset):
    """
    Each __getitem__ returns:
        image   : (3, IMAGE_SIZE, IMAGE_SIZE)  float32 tensor
        caption : (MAX_LENGTH,)                int64 tensor
                  = [<start>, w1, w2, ..., wN, <end>, <pad>, <pad>, ...]

    Why include <start> in the caption tensor?
    The decoder receives captions[:, :-1] as input and captions[:, 1:] as target.
    So input  = [<start>, w1, ..., wN]
       target = [w1,      w2, ..., wN, <end>]
    This is teacher-forced next-word prediction.
    """
    def __init__(self, image_ids, mapping, vocab, image_dir, transform=None):
        self.image_dir = image_dir
        self.vocab     = vocab
        self.transform = transform
        self.samples   = []                           # (image_id, caption_str)

        for img_id in image_ids:
            for caption in mapping.get(img_id, []):
                self.samples.append((img_id, caption))

        print(f'Dataset — {len(self.samples)} (image, caption) pairs')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_id, caption = self.samples[idx]

        # ── Image ──────────────────────────────────────────────────────────────
        img_path = os.path.join(self.image_dir, image_id + '.jpg')
        image    = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # ── Caption ────────────────────────────────────────────────────────────
        tokens  = ['<start>'] + caption.split() + ['<end>']
        tokens  = tokens[:MAX_LENGTH]           # truncate if caption is very long
        encoded = [self.vocab.word2idx.get(w, self.vocab.word2idx['<unk>'])
                   for w in tokens]
        pad_id  = self.vocab.word2idx['<pad>']
        padded  = encoded + [pad_id] * (MAX_LENGTH - len(encoded))

        return image, torch.tensor(padded, dtype=torch.long)


In [ ]:
def get_dataloaders(mapping, vocab, image_dir, train_ids, val_ids, test_ids):
    exists  = lambda img_id: os.path.exists(os.path.join(image_dir, img_id + '.jpg'))
    train_f = [i for i in train_ids if exists(i)]
    val_f   = [i for i in val_ids   if exists(i)]
    test_f  = [i for i in test_ids  if exists(i)]

    train_ds = CaptionDataset(train_f, mapping, vocab, image_dir, get_transforms(True))
    val_ds   = CaptionDataset(val_f,   mapping, vocab, image_dir, get_transforms(False))
    test_ds  = CaptionDataset(test_f,  mapping, vocab, image_dir, get_transforms(False))

    # num_workers=0 on CPU — multiple workers add IPC overhead that slows things down
    # num_workers=2 on GPU — parallel image loading fills the GPU pipeline
    n_workers = 2 if USE_AMP else 0
    kw = dict(num_workers=n_workers, pin_memory=USE_AMP)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  **kw)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **kw)

    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = get_dataloaders(
    mapping, vocab, IMAGE_DIR, train_ids, val_ids, test_ids
)

imgs, caps = next(iter(train_loader))
print(f'Image batch : {imgs.shape}')
print(f'Caption batch: {caps.shape}')


---
## Section 8 — Custom CNN Encoder

**Concept:** The CNN converts an image `(B, 3, 224, 224)` into a fixed-size feature vector `(B, CNN_FEAT_DIM)` that summarises *what is in the image*. We use 5 convolutional blocks, each doubling the number of filters while halving the spatial resolution.

```
Input  (B, 3,   224, 224)
Block1 (B, 32,  112, 112)  ← Conv + BN + ReLU + MaxPool
Block2 (B, 64,   56,  56)
Block3 (B, 128,  28,  28)
Block4 (B, 256,  14,  14)
Block5 (B, 512,  14,  14)  ← no pool — keeps spatial detail
AvgPool(B, 512,   1,   1)
Flatten(B, 512)
Linear (B, 1024) → Dropout → Linear(B, 4096)
```

**Why `bias=False` with BatchNorm?**  
BatchNorm has its own learnable bias parameter (β). Adding a Conv bias would be redundant and waste parameters.

**Bug in original (cell 8):**  
Keras warned: *"Do not pass input_shape to a layer; use an Input() object instead."*  
In PyTorch there is no such concept — shapes are inferred automatically on the first forward pass. No explicit input declaration is needed.

In [ ]:
class CNNBlock(nn.Module):
    """Conv2d → BatchNorm2d → ReLU → [MaxPool2d]"""
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class CNNEncoder(nn.Module):
    """
    5-block CNN that outputs a (B, CNN_FEAT_DIM) feature vector per image.
    The feature vector becomes the initial hidden state of the LSTM decoder.
    """
    def __init__(self, feat_dim=CNN_FEAT_DIM, dropout=DROPOUT):
        super().__init__()
        self.features = nn.Sequential(
            CNNBlock(3,   32),            # 224 → 112
            CNNBlock(32,  64),            # 112 → 56
            CNNBlock(64,  128),           # 56  → 28
            CNNBlock(128, 256),           # 28  → 14
            CNNBlock(256, 512, pool=False),  # 14 → 14 (no pool)
        )
        self.pool    = nn.AdaptiveAvgPool2d((1, 1))  # (B, 512, 14, 14) → (B, 512, 1, 1)
        self.flatten = nn.Flatten()                   # → (B, 512)
        self.proj    = nn.Sequential(
            nn.Linear(512, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(1024, feat_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.features(x)   # (B, 512, 14, 14)
        x = self.pool(x)        # (B, 512,  1,  1)
        x = self.flatten(x)     # (B, 512)
        x = self.proj(x)        # (B, feat_dim)
        return x


---
## Section 9 — LSTM Decoder

**Concept:** The decoder generates one word at a time, conditioned on:
1. The image feature (encoded into the **initial hidden state** of the LSTM)
2. All previously generated words (via the LSTM's own hidden state)

**Teacher forcing (training):** We always feed the *ground-truth* previous word as input, not the model's own prediction. This makes training stable and fast, but creates a small train/inference gap (handled during generation).

**Bug in original (cell 31) — wrong fusion of image and text:**
```python
fe2    = Dense(256)(image_feat)     # image → 256-dim vector
seq3   = LSTM(256)(word_embed)      # LSTM processes words independently
merged = add([fe2, seq3])           # image only fused ONCE, after LSTM
```
The image was added to the LSTM output **after** the LSTM had already processed all words independently. The LSTM had **no image information during word generation** — it only saw image information at the final merge step.

**Fix:** Project the image feature into the **initial hidden state** `(h0, c0)` of the LSTM. This way the image influences **every step** of word generation — the correct approach for image-conditioned language models.

**Beam search (inference):** Instead of always picking the single highest-probability word (greedy), beam search keeps the top-`k` candidate sequences alive at each step, then picks the best complete sequence at the end. This avoids getting stuck in locally-good but globally-bad word choices.

In [ ]:
class LSTMDecoder(nn.Module):
    """
    2-layer LSTM that generates captions word-by-word.
    Image features initialise the hidden state so every generation
    step is conditioned on the image (fixes the original add() bug).
    """
    def __init__(self, vocab_size, embed_dim=EMBED_DIM,
                 lstm_units=LSTM_UNITS, feat_dim=CNN_FEAT_DIM,
                 max_length=MAX_LENGTH, dropout=DROPOUT):
        super().__init__()
        self.max_length = max_length
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.drop       = nn.Dropout(dropout)
        self.img_proj   = nn.Linear(feat_dim, lstm_units)   # image → h0
        self.lstm       = nn.LSTM(embed_dim, lstm_units,
                                  num_layers=2, batch_first=True,
                                  dropout=dropout)
        self.fc         = nn.Linear(lstm_units, vocab_size)

    def forward(self, features, captions):
        """
        Training forward pass (teacher forcing).
        features : (B, feat_dim)   — from CNN
        captions : (B, MAX_LENGTH) — [<start>, w1, ..., wN, <end>, <pad>...]
        Returns logits : (B, MAX_LENGTH-1, vocab_size)
        Target is captions[:, 1:]  i.e. [w1, ..., wN, <end>, <pad>...]
        """
        # Image feature → initial hidden + cell state
        h0 = self.img_proj(features).unsqueeze(0).repeat(2, 1, 1)  # (2, B, lstm_units)
        c0 = torch.zeros_like(h0)

        # Embed all tokens except the last (we never feed <end> as input)
        embeds = self.drop(self.embedding(captions[:, :-1]))  # (B, MAX_LENGTH-1, embed_dim)

        out, _ = self.lstm(embeds, (h0, c0))   # (B, MAX_LENGTH-1, lstm_units)
        logits = self.fc(out)                   # (B, MAX_LENGTH-1, vocab_size)
        return logits

    @torch.no_grad()
    def generate_greedy(self, feature, vocab):
        """
        Greedy decoding — always pick the highest-probability word.

        FIX for original cell 37: vocab.idx2word is a plain dict, so
        each lookup is O(1). The original looped over the entire
        tokenizer.word_index (~8000 entries) for every single word generated.
        """
        self.eval()
        h = self.img_proj(feature).unsqueeze(0).repeat(2, 1, 1)
        c = torch.zeros_like(h)

        word_idx = vocab.word2idx['<start>']
        result   = []

        for _ in range(self.max_length):
            token  = torch.tensor([[word_idx]], device=feature.device)
            embed  = self.drop(self.embedding(token))      # (1, 1, embed_dim)
            out, (h, c) = self.lstm(embed, (h, c))         # (1, 1, lstm_units)
            word_idx = self.fc(out.squeeze(1)).argmax(-1).item()  # O(1)
            word     = vocab.idx2word.get(word_idx, '<unk>')      # O(1)
            if word == '<end>':
                break
            if word not in ('<pad>', '<start>'):
                result.append(word)

        return ' '.join(result)

    @torch.no_grad()
    def generate_beam(self, feature, vocab, beam_width=BEAM_WIDTH):
        """
        Beam search — keep top-k candidate sequences alive at every step.
        Produces noticeably better captions than greedy at low extra cost.
        """
        self.eval()
        h0 = self.img_proj(feature).unsqueeze(0).repeat(2, 1, 1)
        c0 = torch.zeros_like(h0)

        start_idx = vocab.word2idx['<start>']
        end_idx   = vocab.word2idx['<end>']

        # Each beam: (cumulative_log_prob, token_list, h, c)
        beams     = [(0.0, [start_idx], h0, c0)]
        completed = []

        for _ in range(self.max_length):
            if not beams:
                break
            new_beams = []
            for score, tokens, h, c in beams:
                if tokens[-1] == end_idx:
                    completed.append((score, tokens))
                    continue
                last  = torch.tensor([[tokens[-1]]], device=feature.device)
                embed = self.drop(self.embedding(last))         # (1, 1, embed_dim)
                out, (h_new, c_new) = self.lstm(embed, (h, c))
                log_p = F.log_softmax(self.fc(out.squeeze(1)), dim=-1)  # (1, vocab)
                top_vals, top_ids = log_p.topk(beam_width)
                for i in range(beam_width):
                    new_beams.append((
                        score + top_vals[0, i].item(),
                        tokens + [top_ids[0, i].item()],
                        h_new, c_new
                    ))
            beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_width]

        completed.extend(beams)
        best = max(completed, key=lambda x: x[0])
        return vocab.decode(best[1])


---
## Section 10 — Full ImageCaptioner Model

**Concept:** We wrap CNNEncoder and LSTMDecoder into a single `nn.Module`. During training, `forward()` is called with teacher forcing. During inference, `generate()` uses beam search.

The model is saved in **PyTorch's native `.pt` format** (a checkpoint dict containing the model weights, optimizer state, epoch number, and val_loss). This replaces the deprecated `.h5` format used in the original (cell 36 warning: *"HDF5 file format is considered legacy"*).

In [ ]:
class ImageCaptioner(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = CNNEncoder()
        self.decoder = LSTMDecoder(vocab_size)

    def forward(self, images, captions):
        features = self.encoder(images)              # (B, CNN_FEAT_DIM)
        logits   = self.decoder(features, captions)  # (B, MAX_LENGTH-1, vocab_size)
        return logits

    @torch.no_grad()
    def generate(self, image, vocab, method='beam'):
        """image: (1, 3, H, W) tensor on DEVICE"""
        feature = self.encoder(image)
        if method == 'beam':
            return self.decoder.generate_beam(feature, vocab)
        return self.decoder.generate_greedy(feature, vocab)


# ── Instantiate & count parameters ────────────────────────────────────────────
model = ImageCaptioner(vocab_size=len(vocab)).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total:,}')
print(f'Trainable params : {trainable:,}')

# Quick shape sanity-check
dummy_img = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
dummy_cap = torch.randint(0, len(vocab), (2, MAX_LENGTH)).to(DEVICE)
out = model(dummy_img, dummy_cap)
print(f'Output shape     : {out.shape}')  # expected (2, MAX_LENGTH-1, vocab_size)


---
## Section 11 — Training Loop

**Concept:** We minimise cross-entropy loss between the predicted word distribution and the ground-truth next word. `ignore_index=0` tells PyTorch to skip the `<pad>` positions when computing the loss — padding is not a word the model should learn to predict.

**Techniques used:**
- **Gradient clipping** (`max_norm=5`): LSTM gradients can explode during backprop-through-time (BPTT). Clipping limits the gradient norm, keeping training stable.
- **ReduceLROnPlateau**: if `val_loss` doesn't improve for 3 epochs, halve the learning rate. Lets the model make bigger steps early and smaller steps when close to a minimum.
- **Best-model checkpoint**: saves weights only when `val_loss` improves — so we always have the best version even if training diverges later.
- **Early stopping**: if `val_loss` hasn't improved for `PATIENCE=5` epochs, stop training. Prevents wasting hours on overfitting.
- **Mixed precision** (`torch.amp`): on CUDA, stores activations in float16 to halve memory usage and double throughput. Falls back to float32 on CPU.

**Bug in original (cell 34) — test set used as validation:**
```python
val_steps     = len(test) // batch_size      # uses TEST set
val_generator = datagenerator(test, ...)     # should be val!
```
The `val` split created in cell 28 was **never used**. Every epoch, the model was validating and checkpointing based on test-set loss. This means:
1. Test data leaked into the training loop (scheduler, checkpointing)
2. The reported BLEU score was optimistically biased

**Fix:** `val_loader` (built from `val_ids`) is used during training. `test_loader` is only touched in Section 12.

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scaler, use_amp):
    model.train()
    total_loss = 0.0
    _device = next(model.parameters()).device

    for images, captions in tqdm(loader, desc='Train', leave=False):
        images   = images.to(_device)
        captions = captions.to(_device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = model(images, captions)
            target = captions[:, 1:]
            B, seq_len, vocab_size = logits.shape
            loss = criterion(logits.reshape(-1, vocab_size), target.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def val_epoch(model, loader, criterion, use_amp):
    model.eval()
    total_loss = 0.0
    _device = next(model.parameters()).device

    for images, captions in tqdm(loader, desc='Val', leave=False):
        images   = images.to(_device)
        captions = captions.to(_device)

        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = model(images, captions)
            target = captions[:, 1:]
            B, seq_len, vocab_size = logits.shape
            loss = criterion(logits.reshape(-1, vocab_size), target.reshape(-1))

        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
def train(model, train_loader, val_loader, vocab):
    _CKPT_DIR = '/home/nathiskar/image_caption_generator/checkpoints'
    os.makedirs(_CKPT_DIR, exist_ok=True)

    # Define locally — never rely on globals that may not be set
    _device  = next(model.parameters()).device   # wherever the model lives
    _use_amp = _device.type == 'cuda'

    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', factor=0.5, patience=2)
    scaler    = torch.cuda.amp.GradScaler(enabled=_use_amp)

    best_val_loss  = float('inf')
    patience_count = 0
    train_losses, val_losses = [], []

    print(f'Device: {_device} | AMP: {_use_amp} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}\n')

    for epoch in range(1, EPOCHS + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, _use_amp)
        vl_loss = val_epoch(model, val_loader, criterion, _use_amp)

        train_losses.append(tr_loss)
        val_losses.append(vl_loss)
        scheduler.step(vl_loss)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:02d}/{EPOCHS}  train={tr_loss:.4f}  val={vl_loss:.4f}  lr={lr_now:.1e}')

        if vl_loss < best_val_loss:
            best_val_loss  = vl_loss
            patience_count = 0
            torch.save({
                'epoch'          : epoch,
                'model_state'    : model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'val_loss'       : vl_loss,
            }, os.path.join(_CKPT_DIR, 'best_model.pt'))
            print(f'  checkpoint saved (val={vl_loss:.4f})')
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f'\nEarly stopping at epoch {epoch}')
                break

    return train_losses, val_losses


In [ ]:
train_losses, val_losses = train(model, train_loader, val_loader, vocab)


---
## Section 12 — Loss Curve

**Concept:** Plotting train vs. validation loss over epochs reveals three possible states:
- **Underfitting**: both losses are high and still falling — train longer or use a bigger model
- **Good fit**: both losses converge to similar low values
- **Overfitting**: train loss keeps falling but val loss rises — our early stopping catches this

The gap between train and val loss is your regularisation target.

In [ ]:
plt.figure(figsize=(9, 4))
epochs_ran = range(1, len(train_losses) + 1)
plt.plot(epochs_ran, train_losses, 'b-o', ms=4, label='Train loss')
plt.plot(epochs_ran, val_losses,   'r-s', ms=4, label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'loss_curve.png'), dpi=150)
plt.show()
print('Loss curve saved.')


---
## Section 13 — Load Best Checkpoint

**Concept:** We always evaluate the **best checkpoint** (lowest val_loss), not the final epoch. The final epoch's weights may have started overfitting while the checkpoint represents the generalization peak.

**Fix for original (cell 36):** PyTorch's `.pt` checkpoint stores a dict so we can always tell what epoch it came from and what val_loss it had. The original `.h5` file stored only weights with no metadata.

In [ ]:
_CKPT_PATH = '/home/nathiskar/image_caption_generator/checkpoints/best_model.pt'
_device     = next(model.parameters()).device

ckpt = torch.load(_CKPT_PATH, map_location=_device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded best checkpoint — epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")


---
## Section 14 — BLEU Evaluation

**Concept:** BLEU (Bilingual Evaluation Understudy) measures how much of the predicted caption's n-grams appear in the reference captions.
- **BLEU-1**: unigram overlap — checks individual word accuracy
- **BLEU-2/3/4**: bigram/trigram/4-gram overlap — checks fluency and phrase correctness

Typical custom-CNN+LSTM scores on Flickr8k: BLEU-1 ≈ 0.45–0.55, BLEU-4 ≈ 0.07–0.12.

**Bug in original (cell 39) — `startseq`/`endseq` included in BLEU:**
```python
y_pred = y_pred.split()                          # includes 'startseq' at position 0
actual_captions = [c.split() for c in captions]  # also has 'startseq'/'endseq'
```
Both references and hypotheses included `startseq`. Since every pair matched on that token, BLEU-1 was artificially inflated.  
**Fix:** `vocab.decode()` strips `<start>`, `<end>`, and `<pad>` — output is clean words only.

**Bug we introduced in v1 of this notebook — DataLoader gives 1 caption per sample:**

Each Flickr8k image has ~5 captions. The `CaptionDataset` stores `(image_id, caption)` pairs, so the DataLoader yields the same image 5 times with 5 different captions.

Iterating over `test_loader` would:
1. Generate **5 separate predictions** for the same image (wasteful, slow)
2. Compare each prediction against only **1 of the 5 references** (BLEU treats this as if there's only 1 reference — artificially harsh AND biased)

The correct approach (which the original notebook did right): iterate over `test_ids` once per image, generate **one prediction**, and collect **all 5 references** from `mapping`.

**`SmoothingFunction.method1`:** For short captions, higher-order BLEU can be 0.0 when there are no n-gram matches. Method1 adds a small smoothing count to prevent division by zero.

In [ ]:
@torch.no_grad()
def evaluate_bleu(model, test_ids, mapping, vocab, image_dir):
    """
    Compute BLEU-1 through BLEU-4 on the test set.

    Key design: iterate over test_ids (one prediction per unique image),
    and collect ALL reference captions for each image from mapping.
    This is the correct BLEU setup for multi-reference datasets like Flickr8k.
    """
    model.eval()
    transform = get_transforms(train=False)
    smoother  = SmoothingFunction().method1
    actual    = []   # list of [[ref1_words], [ref2_words], ...] per image
    predict   = []   # list of [pred_words] per image

    for img_id in tqdm(test_ids, desc='Evaluating'):
        img_path = os.path.join(image_dir, img_id + '.jpg')
        if not os.path.exists(img_path):
            continue

        # ONE prediction per image
        image_pil    = Image.open(img_path).convert('RGB')
        image_tensor = transform(image_pil).unsqueeze(0).to(DEVICE)
        pred_str     = model.generate(image_tensor, vocab, method='beam')

        # ALL reference captions for this image (mapping is already cleaned)
        refs = [c.split() for c in mapping.get(img_id, [])]
        if not refs:
            continue

        actual.append(refs)               # [[ref1_words], [ref2_words], ...]
        predict.append(pred_str.split())  # [pred_words]

    b1 = corpus_bleu(actual, predict, weights=(1.0, 0, 0, 0),           smoothing_function=smoother)
    b2 = corpus_bleu(actual, predict, weights=(0.5, 0.5, 0, 0),         smoothing_function=smoother)
    b3 = corpus_bleu(actual, predict, weights=(0.33, 0.33, 0.33, 0),    smoothing_function=smoother)
    b4 = corpus_bleu(actual, predict, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoother)

    print(f'Evaluated {len(predict)} images')
    print(f'\nBLEU-1 : {b1:.4f}')
    print(f'BLEU-2 : {b2:.4f}')
    print(f'BLEU-3 : {b3:.4f}')
    print(f'BLEU-4 : {b4:.4f}')
    return b1, b2, b3, b4


# test_loader is NOT used here — iterate per image_id instead
bleu_scores = evaluate_bleu(model, test_ids, mapping, vocab, IMAGE_DIR)


---
## Section 15 — Visual Demo

**Concept:** Qualitative evaluation — look at actual images with their predicted and reference captions. This reveals failure modes that BLEU scores hide: does the model hallucinate objects? Does it describe backgrounds instead of subjects? Is the grammar correct?

In [ ]:
def show_caption(image_id, mapping, vocab, model, image_dir, method='beam'):
    """Display one image with its predicted and reference captions side by side."""
    img_path  = os.path.join(image_dir, image_id + '.jpg')
    image_pil = Image.open(img_path).convert('RGB')

    transform    = get_transforms(train=False)
    image_tensor = transform(image_pil).unsqueeze(0).to(DEVICE)

    predicted = model.generate(image_tensor, vocab, method=method)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(image_pil)
    ax.axis('off')

    refs = '\n'.join(f'  {i+1}. {c}' for i, c in enumerate(mapping[image_id]))
    title = f'Predicted ({method}): {predicted}\n\nReferences:\n{refs}'
    ax.set_title(title, fontsize=9, loc='left')
    plt.tight_layout()
    plt.show()


# Run on the first 5 test images that exist on disk
shown = 0
for img_id in test_ids:
    if shown >= 5:
        break
    path = os.path.join(IMAGE_DIR, img_id + '.jpg')
    if os.path.exists(path):
        show_caption(img_id, mapping, vocab, model, IMAGE_DIR)
        shown += 1


---
## Summary of All Bugs Fixed

| # | Where | Bug | Severity | Fix |
|---|---|---|---|---|
| 1 | Original cell 10 | `preprocess_input` not imported → `NameError` | **Critical** | `transforms.Normalize` in DataLoader |
| 2 | Original cell 18 | `str.replace()` used for regex → punctuation never removed | **Critical** | `re.sub()` |
| 3 | Original cell 34 | Test set used as validation every epoch | **Critical** | `val_loader` (not `test_loader`) in training loop |
| 4 | Original cell 25 | Tokenizer fit on all captions incl. val/test | **Data leakage** | Vocab built from train captions only |
| 5 | Original cell 16 | `split(',')` truncates captions containing commas | **Wrong results** | `split(',', 1)` |
| 6 | Original cell 39 | `startseq`/`endseq` included in BLEU | **Metric inflation** | `vocab.decode()` strips special tokens |
| 7 | Original cell 31 | Image fused with `add()` after LSTM — LSTM gets no image context | **Architecture** | Image projected into LSTM initial hidden state |
| 8 | Original cell 28 | No shuffle before train/val/test split | **Statistical bias** | `random.shuffle(ids)` before slicing |
| 9 | Original cell 37 | `idx_to_word` scans entire vocab — O(V) per word | **Performance** | `vocab.idx2word` dict — O(1) lookup |
| 10 | Original cell 34 | No gradient clipping → exploding LSTM gradients | **Training stability** | `clip_grad_norm_(max_norm=5.0)` |
| 11 | Original cell 34 | No early stopping → wasted compute on overfitting | **Training quality** | `patience=5` early stopping |
| 12 | Original cell 34 | No LR scheduling | **Training quality** | `ReduceLROnPlateau(factor=0.5, patience=3)` |
| 13 | Original cell 36 | Saved in deprecated `.h5` format | **Minor** | PyTorch `.pt` checkpoint dict |
| 14 | This notebook v1 | `evaluate_bleu` iterated DataLoader — 5 preds per image, 1 ref each | **Wrong BLEU** | Iterate `test_ids` directly — 1 pred, all 5 refs per image |


---
---

# VERSION 2 — ResNet50 Encoder + Attention Decoder

**What changes from V1:**

| | Version 1 | Version 2 |
|---|---|---|
| **Encoder** | Custom 5-block CNN (trained from scratch) | ResNet50 pretrained on ImageNet |
| **Image features** | Single global vector `(B, 4096)` | Spatial feature map `(B, 49, 512)` |
| **Decoder** | LSTM with image as initial hidden state | LSTM with **attention** at every step |
| **Word generation** | Same image context for every word | Different image region per word |
| **Beam search** | Raw log-prob score | Length-normalised score |
| **Loss** | CrossEntropy | CrossEntropy + **label smoothing 0.1** |

**Expected BLEU-4 improvement:** ~0.07 → ~0.20+


---
## Section 16 — V2 Encoder: Pretrained ResNet50

**Why ResNet50 instead of custom CNN?**

Your custom CNN was trained on only 8,091 images — far too few to learn rich visual features from scratch. ResNet50 was pretrained on **1.2 million ImageNet images** and already understands textures, shapes, and objects before training even starts.

**Key architectural change — spatial features:**

```
V1 Custom CNN:    (B, 3, 224, 224) → single vector (B, 4096)
                                        ↑ one number summarises the whole image

V2 ResNet50:      (B, 3, 224, 224) → spatial map (B, 49, 512)
                                        ↑ 49 = 7×7 grid of regions, each with 512 features
                                        attention uses this grid to focus per word
```

**Fine-tuning strategy:** We freeze the early ResNet layers (they detect basic edges/textures — already perfect) and only fine-tune the last block (`layer4`). This prevents destroying pretrained knowledge while adapting to our captioning task.


In [ ]:
import torchvision.models as models

V2_FEAT_DIM = 512   # projected spatial feature size (from ResNet's 2048)

class PretrainedEncoder(nn.Module):
    """
    ResNet50 with the final avgpool + fc removed.
    Outputs a spatial feature map (B, 49, V2_FEAT_DIM) instead of a single vector.
    49 = 7x7 spatial grid — each cell represents one region of the image.
    """
    def __init__(self, feat_dim=V2_FEAT_DIM, dropout=DROPOUT):
        super().__init__()
        resnet = models.resnet50(weights='IMAGENET1K_V1')
        # Drop last 2 layers (AdaptiveAvgPool + Linear classifier)
        self.resnet = nn.Sequential(*list(resnet.children())[:-2])  # (B, 2048, 7, 7)

        # Project 2048 → feat_dim to reduce memory and computation
        self.proj = nn.Linear(2048, feat_dim)
        self.drop = nn.Dropout(dropout)

        # Freeze all layers except layer4 (last residual block)
        # Early layers detect edges/textures — already perfect, don't touch
        for name, param in self.resnet.named_parameters():
            param.requires_grad = 'layer4' in name

    def forward(self, x):
        with torch.set_grad_enabled(self.training):
            feat = self.resnet(x)                      # (B, 2048, 7, 7)
        B, C, H, W = feat.shape
        feat = feat.permute(0, 2, 3, 1).reshape(B, H*W, C)  # (B, 49, 2048)
        feat = self.drop(self.proj(feat))              # (B, 49, feat_dim)
        return feat


# Quick shape check
_dummy = torch.randn(2, 3, 224, 224)
_enc   = PretrainedEncoder()
print(f'Encoder output: {_enc(_dummy).shape}')  # expected (2, 49, 512)
del _dummy, _enc


---
## Section 17 — Bahdanau Attention

**The problem with V1:** The image was compressed into one vector fed as the LSTM's initial hidden state. After that, the LSTM generated words using only its own memory — no way to re-look at the image.

**Attention solution:** At every word generation step, the decoder asks: *"Which of the 49 image regions matters most right now?"*

```
Generating "dog":
    attention weights: [0.01, 0.01, 0.70, 0.02, ...]  ← focuses on dog region

Generating "grass":
    attention weights: [0.02, 0.60, 0.01, 0.01, ...]  ← focuses on grass region
```

**How the weights are computed:**

```
features  (B, 49, feat_dim)   ← 49 image regions
hidden    (B, lstm_units)     ← current decoder state (what we've generated so far)

energy = tanh( W_feat(features) + W_hidden(hidden) )   ← score each region
alpha  = softmax(energy)                                ← normalise → probabilities
context = sum(alpha × features)                         ← weighted average region
```

`context` is then concatenated with the word embedding and fed into the LSTM.


In [ ]:
class BahdanauAttention(nn.Module):
    """
    Soft attention over spatial image features.
    Returns context vector (weighted sum of regions) and alpha (attention map).
    """
    def __init__(self, feat_dim, lstm_units, attn_dim=256):
        super().__init__()
        self.W_feat   = nn.Linear(feat_dim,   attn_dim)
        self.W_hidden = nn.Linear(lstm_units, attn_dim)
        self.v        = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, features, hidden):
        # features : (B, 49, feat_dim)
        # hidden   : (B, lstm_units)
        att_feat   = self.W_feat(features)                 # (B, 49, attn_dim)
        att_hidden = self.W_hidden(hidden).unsqueeze(1)    # (B,  1, attn_dim)
        energy     = torch.tanh(att_feat + att_hidden)     # (B, 49, attn_dim)
        alpha      = torch.softmax(self.v(energy), dim=1)  # (B, 49, 1)
        context    = (alpha * features).sum(dim=1)         # (B, feat_dim)
        return context, alpha.squeeze(-1)                  # (B, feat_dim), (B, 49)


class AttentionLSTMDecoder(nn.Module):
    """
    LSTM decoder that re-attends to the image at every word generation step.

    V1 difference: V1 fed image as h0 once. V2 recomputes image context
    at every timestep using attention — the image is consulted 35 times,
    once per word position, with a different focus each time.

    Also adds length normalisation in beam search to prevent short caption bias.
    """
    def __init__(self, vocab_size, feat_dim=V2_FEAT_DIM,
                 embed_dim=EMBED_DIM, lstm_units=LSTM_UNITS,
                 max_length=MAX_LENGTH, dropout=DROPOUT):
        super().__init__()
        self.max_length = max_length

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = BahdanauAttention(feat_dim, lstm_units)
        self.drop      = nn.Dropout(dropout)

        # Initialise h, c from mean-pooled spatial features
        self.init_h = nn.Linear(feat_dim, lstm_units)
        self.init_c = nn.Linear(feat_dim, lstm_units)

        # LSTMCell: input = [embedding || context]
        self.lstm = nn.LSTMCell(embed_dim + feat_dim, lstm_units)
        self.fc   = nn.Linear(lstm_units, vocab_size)

    def _init_hidden(self, features):
        mean = features.mean(dim=1)                  # (B, feat_dim)
        return torch.tanh(self.init_h(mean)), torch.tanh(self.init_c(mean))

    def forward(self, features, captions):
        """Teacher-forcing: step through each token position."""
        h, c    = self._init_hidden(features)
        logits  = []
        seq_len = captions.size(1) - 1              # exclude last token

        for t in range(seq_len):
            context, _  = self.attention(features, h)
            embed       = self.drop(self.embedding(captions[:, t]))
            lstm_in     = torch.cat([embed, context], dim=1)
            h, c        = self.lstm(lstm_in, (h, c))
            logits.append(self.fc(self.drop(h)))

        return torch.stack(logits, dim=1)            # (B, seq_len, vocab_size)

    @torch.no_grad()
    def generate_beam(self, features, vocab, beam_width=BEAM_WIDTH):
        """Beam search with length normalisation — fixes short caption bias."""
        self.eval()
        h, c      = self._init_hidden(features)
        start_idx = vocab.word2idx['<start>']
        end_idx   = vocab.word2idx['<end>']

        beams     = [(0.0, [start_idx], h, c)]
        completed = []

        for _ in range(self.max_length):
            if not beams:
                break
            new_beams = []
            for score, tokens, h, c in beams:
                if tokens[-1] == end_idx:
                    completed.append((score, tokens))
                    continue
                last    = torch.tensor([tokens[-1]], device=features.device)
                embed   = self.drop(self.embedding(last))
                context, _ = self.attention(features, h)
                lstm_in = torch.cat([embed, context], dim=1)
                h_new, c_new = self.lstm(lstm_in, (h, c))
                log_p   = F.log_softmax(self.fc(h_new), dim=-1)
                top_vals, top_ids = log_p.topk(beam_width)
                for i in range(beam_width):
                    new_beams.append((
                        score + top_vals[0, i].item(),
                        tokens + [top_ids[0, i].item()],
                        h_new, c_new
                    ))
            beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_width]

        completed.extend(beams)
        # Length normalisation: divide by sequence length to avoid short bias
        best = max(completed, key=lambda x: x[0] / max(len(x[1]), 1))
        return vocab.decode(best[1])


---
## Section 18 — Full V2 Model + Training


In [ ]:
class ImageCaptionerV2(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = PretrainedEncoder(feat_dim=V2_FEAT_DIM)
        self.decoder = AttentionLSTMDecoder(vocab_size, feat_dim=V2_FEAT_DIM)

    def forward(self, images, captions):
        features = self.encoder(images)              # (B, 49, V2_FEAT_DIM)
        logits   = self.decoder(features, captions)  # (B, seq_len-1, vocab_size)
        return logits

    @torch.no_grad()
    def generate(self, image, vocab, method='beam'):
        features = self.encoder(image)
        return self.decoder.generate_beam(features, vocab)


# ── Instantiate ────────────────────────────────────────────────────────────────
_device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_v2 = ImageCaptionerV2(vocab_size=len(vocab)).to(_device)

total     = sum(p.numel() for p in model_v2.parameters())
trainable = sum(p.numel() for p in model_v2.parameters() if p.requires_grad)
print(f'Total params     : {total:,}')
print(f'Trainable params : {trainable:,}')
print(f'(Frozen ResNet params are excluded from trainable count)')

# Shape sanity check
_img = torch.randn(2, 3, 224, 224).to(_device)
_cap = torch.randint(0, len(vocab), (2, MAX_LENGTH)).to(_device)
print(f'\nOutput shape: {model_v2(_img, _cap).shape}')  # (2, MAX_LENGTH-1, vocab_size)
del _img, _cap


In [ ]:
def train_v2(model, train_loader, val_loader):
    """
    Training loop for V2.
    Differences from V1:
      - label_smoothing=0.1 in CrossEntropyLoss (reduces overconfidence)
      - Lower LR for encoder (pretrained weights are fragile) vs decoder
    """
    _CKPT_DIR = '/home/nathiskar/image_caption_generator/checkpoints'
    os.makedirs(_CKPT_DIR, exist_ok=True)

    _device  = next(model.parameters()).device
    _use_amp = _device.type == 'cuda'

    # Label smoothing — V2 improvement over V1
    criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)

    # Two param groups: encoder gets 10x lower LR to avoid destroying pretrained weights
    optimizer = torch.optim.Adam([
        {'params': model.encoder.parameters(), 'lr': LEARNING_RATE / 10},
        {'params': model.decoder.parameters(), 'lr': LEARNING_RATE},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', factor=0.5, patience=2)
    scaler    = torch.cuda.amp.GradScaler(enabled=_use_amp)

    best_val_loss  = float('inf')
    patience_count = 0
    train_losses, val_losses = [], []

    print(f'V2 Training | Device: {_device} | AMP: {_use_amp}')
    print(f'Encoder LR: {LEARNING_RATE/10:.1e}  Decoder LR: {LEARNING_RATE:.1e}\n')

    for epoch in range(1, EPOCHS + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, _use_amp)
        vl_loss = val_epoch(model, val_loader, criterion, _use_amp)

        train_losses.append(tr_loss)
        val_losses.append(vl_loss)
        scheduler.step(vl_loss)

        lr_dec = optimizer.param_groups[1]['lr']
        print(f'Epoch {epoch:02d}/{EPOCHS}  train={tr_loss:.4f}  val={vl_loss:.4f}  lr={lr_dec:.1e}')

        if vl_loss < best_val_loss:
            best_val_loss  = vl_loss
            patience_count = 0
            torch.save({
                'epoch'      : epoch,
                'model_state': model.state_dict(),
                'val_loss'   : vl_loss,
            }, os.path.join(_CKPT_DIR, 'best_model_v2.pt'))
            print(f'  checkpoint saved (val={vl_loss:.4f})')
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f'\nEarly stopping at epoch {epoch}')
                break

    return train_losses, val_losses


train_losses_v2, val_losses_v2 = train_v2(model_v2, train_loader, val_loader)


---
## Section 19 — V1 vs V2 Comparison


In [ ]:
# ── Re-define evaluate_bleu here so this cell is fully self-contained ─────────
@torch.no_grad()
def evaluate_bleu(model, test_ids, mapping, vocab, image_dir):
    model.eval()
    transform = get_transforms(train=False)
    smoother  = SmoothingFunction().method1
    actual, predict = [], []
    _device = next(model.parameters()).device

    for img_id in tqdm(test_ids, desc='Evaluating', leave=False):
        img_path = os.path.join(image_dir, img_id + '.jpg')
        if not os.path.exists(img_path):
            continue
        image_pil    = Image.open(img_path).convert('RGB')
        image_tensor = transform(image_pil).unsqueeze(0).to(_device)
        pred_str     = model.generate(image_tensor, vocab)
        refs         = [c.split() for c in mapping.get(img_id, [])]
        if not refs:
            continue
        actual.append(refs)
        predict.append(pred_str.split())

    b1 = corpus_bleu(actual, predict, weights=(1.0, 0, 0, 0),           smoothing_function=smoother)
    b2 = corpus_bleu(actual, predict, weights=(0.5, 0.5, 0, 0),         smoothing_function=smoother)
    b3 = corpus_bleu(actual, predict, weights=(0.33, 0.33, 0.33, 0),    smoothing_function=smoother)
    b4 = corpus_bleu(actual, predict, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoother)
    return b1, b2, b3, b4


# ── Loss curve: V1 vs V2 ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(train_losses,    'b-o',  ms=4, label='V1 Train')
axes[0].plot(val_losses,      'b--s', ms=4, label='V1 Val')
axes[0].set_title('Version 1 — Custom CNN + LSTM')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(train_losses_v2, 'r-o',  ms=4, label='V2 Train')
axes[1].plot(val_losses_v2,   'r--s', ms=4, label='V2 Val')
axes[1].set_title('Version 2 — ResNet50 + Attention')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/nathiskar/image_caption_generator/outputs/loss_v1_vs_v2.png', dpi=150)
plt.show()

# ── BLEU: V1 vs V2 ────────────────────────────────────────────────────────────
print('=' * 50)
print('Evaluating V1 ...')
b1_v1, b2_v1, b3_v1, b4_v1 = evaluate_bleu(model, test_ids, mapping, vocab, IMAGE_DIR)

_v2_ckpt = '/home/nathiskar/image_caption_generator/checkpoints/best_model_v2.pt'
_device  = next(model_v2.parameters()).device
model_v2.load_state_dict(torch.load(_v2_ckpt, map_location=_device)['model_state'])
model_v2.eval()

print('Evaluating V2 ...')
b1_v2, b2_v2, b3_v2, b4_v2 = evaluate_bleu(model_v2, test_ids, mapping, vocab, IMAGE_DIR)

print('\n' + '=' * 50)
print(f'{"":12} {"BLEU-1":>8} {"BLEU-2":>8} {"BLEU-3":>8} {"BLEU-4":>8}')
print(f'{"V1 (CNN)":12} {b1_v1:8.4f} {b2_v1:8.4f} {b3_v1:8.4f} {b4_v1:8.4f}')
print(f'{"V2 (ResNet)":12} {b1_v2:8.4f} {b2_v2:8.4f} {b3_v2:8.4f} {b4_v2:8.4f}')
print(f'{"Gain":12} {b1_v2-b1_v1:+8.4f} {b2_v2-b2_v1:+8.4f} {b3_v2-b3_v1:+8.4f} {b4_v2-b4_v1:+8.4f}')
print('=' * 50)

# ── Side-by-side caption demo ─────────────────────────────────────────────────
_transform = get_transforms(train=False)
print('\n--- Caption comparison on 3 test images ---')
for img_id in test_ids[:3]:
    _path = os.path.join(IMAGE_DIR, img_id + '.jpg')
    if not os.path.exists(_path):
        continue
    _pil    = Image.open(_path).convert('RGB')
    _tensor = _transform(_pil).unsqueeze(0).to(_device)
    print(f'\nImage : {img_id}')
    print(f'  V1  : {model.generate(_tensor.to(next(model.parameters()).device), vocab)}')
    print(f'  V2  : {model_v2.generate(_tensor, vocab)}')
    print(f'  Ref : {mapping[img_id][0]}')


---
## Section 20 — Test Any Image with V2


In [ ]:
def compare_captions(image_path, model_v1, model_v2, vocab):
    """
    Show V1 and V2 captions side by side on the same image.
    Left panel = V1,  Right panel = V2.
    """
    _CKPT_V1 = '/home/nathiskar/image_caption_generator/checkpoints/best_model.pt'
    _CKPT_V2 = '/home/nathiskar/image_caption_generator/checkpoints/best_model_v2.pt'
    _dev_v1  = next(model_v1.parameters()).device
    _dev_v2  = next(model_v2.parameters()).device

    # Load best weights for both models
    model_v1.load_state_dict(torch.load(_CKPT_V1, map_location=_dev_v1)['model_state'])
    model_v2.load_state_dict(torch.load(_CKPT_V2, map_location=_dev_v2)['model_state'])
    model_v1.eval()
    model_v2.eval()

    # Preprocess
    transform    = get_transforms(train=False)
    image_pil    = Image.open(image_path).convert('RGB')
    tensor_v1    = transform(image_pil).unsqueeze(0).to(_dev_v1)
    tensor_v2    = transform(image_pil).unsqueeze(0).to(_dev_v2)

    # Generate captions
    with torch.no_grad():
        cap_v1 = model_v1.generate(tensor_v1, vocab)
        cap_v2 = model_v2.generate(tensor_v2, vocab)

    # Plot — same image twice, different caption on each panel
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax in axes:
        ax.imshow(image_pil)
        ax.axis('off')

    axes[0].set_title(f'V1  (Custom CNN + LSTM)\n\n"{cap_v1}"',
                      fontsize=11, color='steelblue', pad=12)
    axes[1].set_title(f'V2  (ResNet50 + Attention)\n\n"{cap_v2}"',
                      fontsize=11, color='crimson',   pad=12)

    plt.suptitle(image_path.split('/')[-1], fontsize=9, color='gray')
    plt.tight_layout()
    plt.savefig('/home/nathiskar/image_caption_generator/outputs/v1_vs_v2_caption.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f'V1 : {cap_v1}')
    print(f'V2 : {cap_v2}')


# ── Change this to any image ───────────────────────────────────────────────────
IMAGE_TO_TEST = '/home/nathiskar/image_caption_generator/data/images/Image/1000268201_693b08cb0e.jpg'

compare_captions(IMAGE_TO_TEST, model, model_v2, vocab)
